# 07 · Web Sitesi Verisi
**Tez:** Güneş Döngüleri 23–25 Boyunca Güneş Patlamaları, Güneş Lekeleri ve Jeomanyetik Etkiler
**Hazırlayan:** Tunahan İnci · **Defter sürümü:** 1.0 (05.11.2026)

Sitedeki tüm grafik ve sayıları 01–06 defterlerinin çıktılarından üretip tek bir `tez_verisi.json` dosyasına yazar. Yeni hesap yapmaz; yalnızca üst üste bindirilmiş olay eğrilerini ve büyük fırtınaların kaynak patlamalarını sitede gösterilecek biçimde yeniden derler.

**Çalıştırma:** Tümünü çalıştır. Sonunda dosya Drive'a (`gunes_tezi/web/tez_verisi.json`) kaydedilir ve tarayıcı otomatik olarak indirir. **İndirilen dosyayı sohbete yükle.**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
KOK = Path('/content/drive/MyDrive/gunes_tezi')

In [ ]:
# Saf fonksiyonlar (test edilebilir)
import numpy as np, pandas as pd

def kumelenmeyi_ayikla(zaman, aki, pencere_saat=48):
    # Büyükten küçüğe giderek, kabul edilmiş bir olaya ±pencere içinde olanları atar.
    # Böylece aynı aktif dönemdeki ardışık patlamalar tek olay sayılır.
    sira = np.argsort(-np.asarray(aki))
    t = pd.to_datetime(pd.Series(zaman)).values.astype('datetime64[s]').astype(np.int64)
    kabul, kabul_t = [], []
    pen = pencere_saat * 3600
    for i in sira:
        if all(abs(t[i] - k) > pen for k in kabul_t):
            kabul.append(i); kabul_t.append(t[i])
    return np.sort(np.array(kabul, dtype=int))

def olay_matrisi(seri, zamanlar, once_saat=48, sonra_saat=120):
    # Saatlik seriden, her olay zamanının etrafındaki pencereyi satır olarak çıkarır.
    s = seri.sort_index()
    t0 = s.index[0]
    deger = s.values.astype(float)
    konum = ((pd.to_datetime(pd.Series(zamanlar)).dt.floor('h') - t0) / pd.Timedelta(hours=1)).values.astype(int)
    gecikme = np.arange(-once_saat, sonra_saat + 1)
    idx = konum[:, None] + gecikme[None, :]
    gecerli = (idx >= 0) & (idx < len(deger))
    M = np.full(idx.shape, np.nan)
    M[gecerli] = deger[idx[gecerli]]
    return gecikme, M

def rastgele_esle(zamanlar, alt, ust, gun=180, tekrar=1, tohum=0):
    # Her olay için ±gun içinde rastgele bir zaman seçer (güneş döngüsü evresini korur).
    rng = np.random.default_rng(tohum)
    z = pd.to_datetime(pd.Series(zamanlar)).values
    out = []
    for _ in range(tekrar):
        kayma = rng.uniform(-gun, gun, len(z)) * 86400
        r = z + (kayma * 1e9).astype('timedelta64[ns]')
        r = np.clip(r, np.datetime64(alt), np.datetime64(ust))
        out.append(pd.DatetimeIndex(r))
    return out

def firtina_var_mi(dst_saatlik, zamanlar, esik=-50, bas_saat=12, son_saat=96):
    gec, M = olay_matrisi(dst_saatlik, zamanlar, 0, son_saat)
    pencere = M[:, bas_saat:]
    return np.nanmin(pencere, axis=1) <= esik

In [ ]:
import json, datetime as dt
ISL = KOK/'data'/'processed'
(KOK/'web').mkdir(parents=True, exist_ok=True)

def yuvarla(x, n=3):
    if x is None: return None
    try:
        if np.isnan(x): return None
    except TypeError:
        pass
    return round(float(x), n)

def liste(seri, n=3):
    return [yuvarla(v, n) for v in seri]

fl = pd.read_parquet(ISL/'flare_final.parquet')
hek = pd.read_parquet(ISL/'flare_hek_temiz.parquet')
sn_ay = pd.read_parquet(ISL/'sn_aylik_1749.parquet').set_index('tarih').sn
sn_dz = pd.read_parquet(ISL/'sn_aylik_duzlestirilmis.parquet').set_index('tarih').sn
omni = pd.read_parquet(ISL/'omni2_saatlik.parquet').set_index('zaman').sort_index()
omni = omni[~omni.index.duplicated()].asfreq('h')
gunluk = pd.read_parquet(ISL/'omni2_gunluk.parquet').set_index('zaman')
V = {}

# Aylık seriler
ay = pd.date_range('2000-01-01', '2025-12-01', freq='MS')
mx_final = fl[fl.sinif.isin(['M', 'X'])].set_index('pik').resample('MS').size().reindex(ay, fill_value=0)
x_final = fl[fl.sinif == 'X'].set_index('pik').resample('MS').size().reindex(ay, fill_value=0)
mx_hek = hek[hek.sinif.isin(['M', 'X'])].set_index('event_peaktime').resample('MS').size().reindex(ay, fill_value=0)
V['aylik'] = {
    'tarih': [d.strftime('%Y-%m') for d in ay],
    'sn': liste(sn_ay.reindex(ay), 1), 'sn_duz': liste(sn_dz.reindex(ay), 1),
    'mx': mx_final.astype(int).tolist(), 'x': x_final.astype(int).tolist(), 'mx_hek': mx_hek.astype(int).tolist(),
    'dst_min': liste(gunluk.dst_min.resample('MS').min().reindex(ay), 0),
    'firtina50': (gunluk.dst_min <= -50).resample('MS').sum().reindex(ay, fill_value=0).astype(int).tolist(),
    'firtina100': (gunluk.dst_min <= -100).resample('MS').sum().reindex(ay, fill_value=0).astype(int).tolist(),
}

# Hazır tablolar
def tablo(dosya, **kw):
    d = pd.read_csv(ISL/dosya, **kw)
    return {'sutunlar': list(d.columns), 'satirlar': [[yuvarla(v, 4) if isinstance(v, (int, float, np.number)) else v for v in r] for r in d.itertuples(index=False)]}
V['dongu_s4'] = tablo('06_dongu_final_S4.csv')
V['dongu_asamalar'] = tablo('03_dongu_ozeti_kalibre.csv')
V['senaryolar'] = tablo('05_dongu23_senaryolar.csv')
V['kalibrasyon'] = tablo('03_kalibrasyon_katsayilari.csv')
V['yillik'] = tablo('06_yillik.csv')
V['dongu_firtina'] = tablo('06_dongu_firtina.csv')
V['olasilik'] = tablo('06_firtina_olasiligi.csv')

# Ölçek farkı: yıllık NCEI/HEK oranı
es = pd.read_parquet(ISL/'ncei_hek_eslesme.parquet').dropna(subset=['oran'])
yo = es.groupby(es.pik.dt.year).oran.agg(medyan='median', q25=lambda s: s.quantile(.25), q75=lambda s: s.quantile(.75), n='size')
V['olcek_orani'] = {'yil': yo.index.astype(int).tolist(), 'medyan': liste(yo.medyan, 3), 'q25': liste(yo.q25, 3), 'q75': liste(yo.q75, 3), 'n': yo.n.astype(int).tolist()}
print('Tablolar ve aylık seriler hazır')

In [ ]:
# Üst üste bindirilmiş olay eğrileri (Dst), 06 ile aynı yöntem
ALT, UST = '2000-01-03', '2025-12-25'
SINIF = {'M1–M5': (1e-5, 5e-5), 'M5–X1': (5e-5, 1e-4), '≥X1': (1e-4, 1)}
V['sea'] = {}
for i, (ad, (a, b)) in enumerate(SINIF.items()):
    s = fl[(fl.aki >= a) & (fl.aki < b) & (fl.pik >= ALT) & (fl.pik <= UST)]
    o = s.iloc[kumelenmeyi_ayikla(s.pik, s.aki, 48)]
    g, M = olay_matrisi(omni.dst, o.pik)
    rast = [np.nanmedian(olay_matrisi(omni.dst, r)[1], 0) for r in rastgele_esle(o.pik, ALT, UST, 180, 200, i)]
    rl, ru = np.nanpercentile(rast, [2.5, 97.5], 0)
    q1, q3 = np.nanpercentile(M, [25, 75], 0)
    sec = slice(None, None, 3)   # 3 saatte bir nokta yeterli
    V['sea'][ad] = {'n': int(len(o)), 'saat': g[sec].tolist(), 'medyan': liste(np.nanmedian(M, 0)[sec], 1),
                    'q1': liste(q1[sec], 1), 'q3': liste(q3[sec], 1), 'rast_alt': liste(rl[sec], 1), 'rast_ust': liste(ru[sec], 1)}
    print(ad, len(o))

# Gecikme analizi
aylik = pd.DataFrame({'SN': sn_ay['2000':'2025'],
                      'M+X': fl[fl.sinif.isin(['M', 'X'])].set_index('pik').resample('MS').size(),
                      'Dst≤−50 gün': (gunluk.dst_min <= -50).resample('MS').sum(),
                      'Dst≤−100 gün': (gunluk.dst_min <= -100).resample('MS').sum()}).fillna(0)['2000':'2025']
duz = aylik.rolling(13, center=True).mean().dropna()
L = list(range(-48, 49))
V['gecikme'] = {'ay': L, **{c: [yuvarla(duz.SN.corr(duz[c].shift(-k)), 3) for k in L] for c in ['M+X', 'Dst≤−50 gün', 'Dst≤−100 gün']}}

In [ ]:
# X sınıfı patlamalar (3B Güneş üzerinde gösterilecek) ve sonrasındaki en düşük Dst
xs = fl[fl.sinif == 'X'].sort_values('pik').copy()
_, Mx = olay_matrisi(omni.dst, xs.pik, 0, 96)
xs['sonra_dst'] = np.nanmin(Mx[:, 12:], axis=1)
V['x_patlamalar'] = [{'t': r.pik.strftime('%Y-%m-%d %H:%M'), 'sinif': r.sinif_metin, 'aki': float(f'{r.aki:.3e}'),
                      'x': yuvarla(r.hpc_x, 0), 'y': yuvarla(r.hpc_y, 0), 'bolge': r.bolge,
                      'ar': int(r.ar_noaanum) if pd.notna(r.ar_noaanum) and r.ar_noaanum > 0 else None,
                      'doymus': bool(r.doymus), 'hudson': None if pd.isna(r.hudson2023_aki) else float(r.hudson2023_aki),
                      'dst': yuvarla(r.sonra_dst, 0)} for r in xs.itertuples()]

# En büyük 10 fırtına günü ve önceki 1–4 günün en büyük patlaması
gf = gunluk.dst_min.dropna()
secilen = []
for t, v in gf.sort_values().items():
    if all(abs((t - s).days) > 5 for s, _ in secilen):
        secilen.append((t, v))
    if len(secilen) == 10: break
firtinalar = []
for t, v in secilen:
    adaylar = fl[(fl.pik >= t - pd.Timedelta(days=4)) & (fl.pik <= t) & (fl.aki >= 1e-6)]
    en = adaylar.loc[adaylar.aki.idxmax()] if len(adaylar) else None
    firtinalar.append({'gun': t.strftime('%Y-%m-%d'), 'dst': int(v),
                       'kp': yuvarla(gunluk.kp_max.get(t), 1),
                       'kaynak_t': None if en is None else en.pik.strftime('%Y-%m-%d %H:%M'),
                       'kaynak_sinif': None if en is None else en.sinif_metin,
                       'kaynak_bolge': None if en is None else en.bolge})
V['en_buyuk_firtinalar'] = firtinalar
print(pd.DataFrame(firtinalar).to_string(index=False))

# En büyük 12 patlama
V['en_buyuk_patlamalar'] = [p for p in sorted(V['x_patlamalar'], key=lambda p: -p['aki'])[:12]]

# Vaka pencereleri
def vaka(bas, son):
    o = omni[bas:son]
    f = fl[(fl.pik >= bas) & (fl.pik <= son) & (fl.aki >= 1e-6)]
    return {'zaman': [t.strftime('%Y-%m-%d %H:%M') for t in o.index], 'dst': liste(o.dst, 0), 'kp': liste(o.kp, 1),
            'patlamalar': [{'t': r.pik.strftime('%Y-%m-%d %H:%M'), 'aki': float(f'{r.aki:.3e}'), 'sinif': r.sinif_metin, 'bolge': r.bolge}
                           for r in f.itertuples()]}
V['vaka'] = {'2003': vaka('2003-10-25', '2003-11-08'), '2024': vaka('2024-05-05', '2024-05-16')}

V['meta'] = {'uretim': dt.datetime.now(dt.timezone.utc).strftime('%Y-%m-%d %H:%M UTC'),
             'toplam_patlama': int(len(fl)), 'sinif_sayilari': fl.sinif.value_counts().to_dict(),
             'donem': '2000-01-01 – 2025-12-31'}

yol = KOK/'web'/'tez_verisi.json'
yol.write_text(json.dumps(V, ensure_ascii=False, separators=(',', ':')))
print(f'\nKaydedildi: {yol}  ({yol.stat().st_size/1024:.0f} KB)')

In [ ]:
from google.colab import files
files.download(str(KOK/'web'/'tez_verisi.json'))